# FOLIO Shared Config + Login

This notebook holds the FOLIO connection settings (Okapi URL, tenant, username) and the login logic, so other notebooks don't each need their own copy.

**How to use it from another notebook**, in the same folder:

```pythen
%run folio_auth.ipynb
```


## Configuration

Edit these three values for your FOLIO tenant. Any notebook that `%run`s this one will inherit them as `OKAPI_URL`, `TENANT`, `USERNAME`.

In [ ]:
import requests
import getpass
import os

In [ ]:
# --- EDIT THESE THREE VALUES ---
OKAPI_URL   = "https://api-example.folio.ebsco.com"     # Your FOLIO API gateway URL (no trailing slash)
TENANT      = "fs00000000"                              # Your FOLIO tenant ID
USERNAME    = "myLoginName"                             # Your FOLIO username
PASSWORD    = os.environ["FOLIO_PASSWORD"]              # Storing your password as an environment variable is more secure than listing it here. This script will also prompt 
                                                        # you for your  password if it is not found in the FOLIO_PASSWORD environment variable.

In [3]:
class FolioAuthError(Exception):
    """Raised when FOLIO login fails or the expected token cookie is missing."""
    pass


def folio_login(okapi_url, tenant, username, password=None):
    """
    Log in to FOLIO via the cookie-based /authn/login-with-expiry endpoint.

    Returns (session, token):
        session : requests.Session with the auth cookie attached and
                  X-Okapi-Tenant / X-Okapi-Token set as default headers
        token   : the raw access token string

    Raises FolioAuthError if login fails or no folioAccessToken cookie
    is found in the response.
    """
    if password is None:
        password = getpass.getpass(f"FOLIO password for {username}: ")

    session = requests.Session()

    login_url = f"{okapi_url}/authn/login-with-expiry"
    headers = {
        "X-Okapi-Tenant": tenant,
        "Content-Type": "application/json",
    }
    payload = {
        "username": username,
        "password": password,
    }

    response = session.post(login_url, headers=headers, json=payload)

    if response.status_code not in (200, 201):
        raise FolioAuthError(
            f"Login failed with status {response.status_code}: {response.text}"
        )

    token = session.cookies.get("folioAccessToken")

    if not token:
        raise FolioAuthError(
            "Login returned a success status but no 'folioAccessToken' cookie "
            f"was found. Cookies received: {session.cookies.get_dict()}. "
            "Your tenant may use a different cookie name -- check with your "
            "FOLIO admin or the response above."
        )

    session.headers.update({
        "X-Okapi-Tenant": tenant,
        "X-Okapi-Token": token,
    })

    return session, token

## Log in

Runs automatically whenever this notebook is executed or `%run` from elsewhere. You'll be prompted for your password.

In [4]:
try:
    session, token = folio_login(
        okapi_url=OKAPI_URL,
        tenant=TENANT,
        username=USERNAME,
        password=PASSWORD
    )
    print("Login succeeded. Token retrieved.")
except FolioAuthError as e:
    session, token = None, None
    print(f"Login failed: {e}")

Login succeeded. Token retrieved.
